In [27]:
import pandas as pd
import numpy as np

file_path = "C:\\Users\\DRUPAKUL\\OneDrive - Otto Group\\Documents\\GitHub\\Own Repo\\Recommendation_Engine\\data\\Crate_and_barrel_sample_data.xlsx" 
df = pd.read_excel(file_path, sheet_name="Customer_Product_Events")
df.columns = [col.strip() for col in df.columns]

# 1.Show the first 3 rows
df.head(3)

,Event_ID,Customer_ID,Name,Email,Phone,Product_ID,Variant_ID,Product_Name,Product_Category,Product_Sub_Category,...,Event_Date,Channel,Session_ID,Sale_Type,Appointment_ID,Design_Room_Type,Is_Custom_Order,Designer_ID,Support_Reason,Related_Order_ID
0,EVT000002,CUST1286,Ishaan Rao,ishaanrao170@sample.com,+91-64002241,P5123,P5123-V1,Scandinavian Black Storage Cabinet,Storage,Shelving,...,2025-09-02,Web,SESS-5L8A78E9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EVT000003,CUST1104,Sandeep Chatterjee,sandeepchatterjee245@sample.com,+91-81084522,P5026,P5026-V1,Industrial Sage Green Hand Towel,Towels,Bath Towels,...,2026-07-03,Web,SESS-T1NTMJSR,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EVT000004,CUST1189,Krishna Nair,krishnanair393@sample.com,+91-75519695,P5085,P5085-V1,Scandinavian Mustard Dining Chair,Furniture,Dining Tables,...,2026-04-01,Web,SESS-UHFJ7UPF,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
#3. Find the shape first
print(df.shape)
# 4.Cardinality of key identifiers
for col in ["Customer_ID", "Product_ID", "Event_ID"]:
        print(f"Unique {col}: {df[col].nunique()}")


(5000, 42)
Unique Customer_ID: 306
Unique Product_ID: 138
Unique Event_ID: 5000


In [29]:
# 6. Understand what an "event" represents - this drives whether we can do collaborative filtering
print("Action value counts:\n", df["Action"].value_counts(dropna=False))
print("\nSale_Type value counts:\n", df["Sale_Type"].value_counts(dropna=False))
print("\nEvent_Date range:", df["Event_Date"].min(), "to", df["Event_Date"].max())

# 7. Avg events per customer / product - tells us how sparse the interaction matrix is
print("\nEvents per customer (describe):\n", df.groupby("Customer_ID").size().describe())
print("\nEvents per product (describe):\n", df.groupby("Product_ID").size().describe())


Action value counts:
 Action
Browsed             1505
Bought              1033
Added_to_Cart        744
Wishlisted           518
Reviewed             398
Support_Contact      249
Returned             189
Design_Booked        144
Design_Completed     143
Design_Converted      77
Name: count, dtype: int64

Sale_Type value counts:
 Sale_Type
NaN             4697
Summer_Sale       78
Winter_Sale       78
Festive_Sale      77
Clearance         70
Name: count, dtype: int64

Event_Date range: 2025-04-11 to 2026-08-24

Events per customer (describe):
 count    306.000000
mean      16.339869
std        4.941843
min        5.000000
25%       13.000000
50%       16.000000
75%       19.000000
max       31.000000
dtype: float64

Events per product (describe):
 count    138.000000
mean      33.572464
std        7.374139
min       20.000000
25%       28.000000
50%       32.500000
75%       38.000000
max       55.000000
dtype: float64


## Feature Engineering: Interactions + Catalog

Split the raw event log into two clean tables:
- **catalog**: one row per `Product_ID` with its static attributes (name, category, brand, price, ...)
- **interactions**: one row per `(Customer_ID, Product_ID)` with a weighted implicit-feedback score built from the `Action` column, plus the most recent event date


In [30]:
# 8. Build the product catalog: one row per Product_ID, dropping rows with no product info
catalog_cols = [
    "Product_ID", "Product_Name", "Product_Category", "Product_Sub_Category",
    "Room_Department", "Brand_Name", "Product_Type", "Color",
    "Material", "Style", "Pattern", "Occasion_Tag", "Product_Price", "Cost",
]

catalog = (
    df.dropna(subset=["Product_ID"])
    .sort_values("Event_Date")
    .drop_duplicates(subset="Product_ID", keep="last")[catalog_cols]
    .reset_index(drop=True)
)

print("Catalog shape:", catalog.shape)
catalog.head(3)


Catalog shape: (138, 14)


,Product_ID,Product_Name,Product_Category,Product_Sub_Category,Room_Department,Brand_Name,Product_Type,Color,Material,Style,Pattern,Occasion_Tag,Product_Price,Cost
0,P5089,Traditional Terracotta Sculpture,Decor,Vases,Decor,Barrel & Co,Sculpture,Terracotta,Ceramic,Traditional,Solid,NaN,1593.12,776.98
1,P5023,Farmhouse Charcoal Hand Towel,Towels,Bath Towels,Bath,Aspen Living,Hand Towel,Charcoal,Cotton,Farmhouse,Solid,Housewarming,1088.21,583.44
2,P5102,Bohemian Terracotta Pendant Light,Lighting,Table Lamps,Decor,Heritage House,Pendant Light,Terracotta,Metal,Bohemian,Solid,NaN,1914.54,803.97


In [31]:
# 9. Weight each Action by strength of implicit signal (tune later based on business input)
ACTION_WEIGHTS = {
    "Bought": 5,
    "Wishlisted": 2,
    "Added_to_Cart": 3,
    "Reviewed": 2,
    "Browsed": 1,
    "Exchanged": 1,      # still a completed purchase, just a size swap
    "Returned": -2,      # negative signal on that product
    "Support_Contact": 0,  # not a preference signal on its own
}

events = df.dropna(subset=["Customer_ID", "Product_ID"]).copy()
events["weight"] = events["Action"].map(ACTION_WEIGHTS).fillna(0)

In [32]:
# Boost/penalize using Rating when available (Reviewed events)
has_rating = events["Rating"].notna()
events.loc[has_rating, "weight"] += (events.loc[has_rating, "Rating"] - 3)  # center around neutral rating=3
#Its a way to make sure that the person's preference is reflected more accurately based on their rating

print("Rows before aggregation:", len(events))
events[["Customer_ID", "Product_ID", "Action", "Rating", "weight", "Event_Date"]].head(5)


Rows before aggregation: 4633


,Customer_ID,Product_ID,Action,Rating,weight,Event_Date
0,CUST1286,P5123,Browsed,NaN,1.0,2025-09-02
1,CUST1104,P5026,Added_to_Cart,NaN,3.0,2026-07-03
2,CUST1189,P5085,Added_to_Cart,NaN,3.0,2026-04-01
3,CUST1125,P5091,Bought,NaN,5.0,2025-07-22
5,CUST1077,P5106,Browsed,NaN,1.0,2026-08-16


In [33]:
# 10. Aggregate to one row per (Customer_ID, Product_ID): sum weights, keep most recent event date
interactions = (
    events.groupby(["Customer_ID", "Product_ID"], as_index=False)
    .agg(
        score=("weight", "sum"),
        n_events=("Action", "count"),
        last_event_date=("Event_Date", "max"),
        actions=("Action", lambda a: sorted(set(a.dropna()))),
    )
)

print("Interactions shape:", interactions.shape)
print("\nScore distribution:\n", interactions["score"].describe())
interactions.sort_values("score", ascending=False).head(5)
print(interactions.head(3))

Interactions shape: (3682, 6)

Score distribution:
 count    3682.000000
mean        2.854155
std         2.268016
min        -2.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        24.000000
Name: score, dtype: float64
  Customer_ID Product_ID  score  n_events last_event_date          actions
0    CUST1000      P5020    1.0         1      2025-09-16        [Browsed]
1    CUST1000      P5051    3.0         1      2026-04-15  [Added_to_Cart]
2    CUST1000      P5074    1.0         1      2026-03-08        [Browsed]


In [34]:
# 11. Sparsity check: how full is the customer x product interaction matrix?
n_customers = interactions["Customer_ID"].nunique()
n_products = interactions["Product_ID"].nunique()
n_filled = len(interactions)
sparsity = 1 - (n_filled / (n_customers * n_products))

print(f"Matrix size: {n_customers} customers x {n_products} products = {n_customers * n_products} cells")
print(f"Filled cells: {n_filled}")
print(f"Sparsity: {sparsity:.2%}")


Matrix size: 306 customers x 138 products = 42228 cells
Filled cells: 3682
Sparsity: 91.28%


In [35]:
# 12. Persist the cleaned tables so the API/training scripts don't need to re-parse the Excel file
import os

os.makedirs("data/processed", exist_ok=True)
catalog.to_csv("data/processed/catalog.csv", index=False)
interactions.to_csv("data/processed/interactions.csv", index=False)

print("Saved data/processed/catalog.csv:", catalog.shape)
print("Saved data/processed/interactions.csv:", interactions.shape)


Saved data/processed/catalog.csv: (138, 14)
Saved data/processed/interactions.csv: (3682, 6)


## Baseline Model: Content Similarity + Popularity

For the MVP baseline (no training required), for a given customer:
1. Build a **customer profile** = weighted average of the attribute vectors of products they've positively interacted with.
2. Score **all catalog products** by cosine similarity to that profile.
3. Blend in a small **popularity** term (overall interaction score) to help break ties and handle thin profiles.
4. Exclude products the customer already bought, and fall back to pure popularity for cold-start customers (no interactions).


In [36]:
# 13. Build a content feature matrix for the catalog (one-hot categorical attrs + scaled price)
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer

CONTENT_CATEGORICAL = [
    "Product_Category", "Product_Sub_Category",
    "Room_Department", "Brand_Name", "Product_Type", "Color",
    "Material", "Style", "Pattern", "Occasion_Tag", 
]

CONTENT_NUMERIC = ["Product_Price"]

catalog_features = catalog.set_index("Product_ID")

content_transformer = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CONTENT_CATEGORICAL),
        ("num", MinMaxScaler(), CONTENT_NUMERIC),
    ]
)

content_matrix = content_transformer.fit_transform(catalog_features)
content_matrix = np.asarray(content_matrix.todense()) if hasattr(content_matrix, "todense") else content_matrix
product_ids = catalog_features.index.to_numpy()

print("Content feature matrix shape:", content_matrix.shape)

Content feature matrix shape: (138, 168)


In [37]:
# 14. Popularity score per product = total interaction score across all customers (min-max scaled to 0-1)
product_popularity = interactions.groupby("Product_ID")["score"].sum()
product_popularity = product_popularity.reindex(product_ids).fillna(0)
pop_min, pop_max = product_popularity.min(), product_popularity.max()
product_popularity_scaled = (product_popularity - pop_min) / (pop_max - pop_min)

product_id_to_idx = {pid: i for i, pid in enumerate(product_ids)}

print(product_popularity_scaled.sort_values(ascending=False).head(5))
print(product_id_to_idx)

Product_ID
P5042    1.000000
P5077    0.951220
P5063    0.926829
P5027    0.853659
P5083    0.841463
Name: score, dtype: float64
{'P5089': 0, 'P5023': 1, 'P5102': 2, 'P5008': 3, 'P5064': 4, 'P5056': 5, 'P5070': 6, 'P5088': 7, 'P5097': 8, 'P5135': 9, 'P5032': 10, 'P5026': 11, 'P5075': 12, 'P5100': 13, 'P5131': 14, 'P5062': 15, 'P5007': 16, 'P5046': 17, 'P5069': 18, 'P5098': 19, 'P5014': 20, 'P5092': 21, 'P5066': 22, 'P5052': 23, 'P5120': 24, 'P5010': 25, 'P5020': 26, 'P5093': 27, 'P5017': 28, 'P5006': 29, 'P5077': 30, 'P5013': 31, 'P5109': 32, 'P5073': 33, 'P5019': 34, 'P5051': 35, 'P5041': 36, 'P5121': 37, 'P5003': 38, 'P5137': 39, 'P5112': 40, 'P5011': 41, 'P5125': 42, 'P5049': 43, 'P5060': 44, 'P5061': 45, 'P5132': 46, 'P5107': 47, 'P5038': 48, 'P5072': 49, 'P5055': 50, 'P5039': 51, 'P5036': 52, 'P5136': 53, 'P5071': 54, 'P5016': 55, 'P5058': 56, 'P5004': 57, 'P5128': 58, 'P5090': 59, 'P5040': 60, 'P5065': 61, 'P5048': 62, 'P5118': 63, 'P5044': 64, 'P5094': 65, 'P5096': 66, 'P5099': 

In [38]:
# 15. Baseline recommender: content similarity to the customer's positive interactions + popularity fallback
from sklearn.metrics.pairwise import cosine_similarity

CONTENT_WEIGHT = 0.7
POPULARITY_WEIGHT = 0.3


def recommend_for_customer(customer_id: str, top_n: int = 5) -> pd.DataFrame:
    customer_rows = interactions[interactions["Customer_ID"] == customer_id]
    positive = customer_rows[customer_rows["score"] > 0]

    seen_products = set(customer_rows["Product_ID"])
    candidate_mask = ~np.isin(product_ids, list(seen_products))

    if positive.empty:
        # Cold-start / no positive signal yet: fall back to pure popularity
        scores = product_popularity_scaled.to_numpy()
        reason = "Popular right now"
    else:
        idxs = positive["Product_ID"].map(product_id_to_idx).dropna().astype(int).to_numpy()
        weights = positive.loc[positive["Product_ID"].isin(product_id_to_idx), "score"].to_numpy()
        weights = weights / weights.sum()

        profile_vector = np.average(content_matrix[idxs], axis=0, weights=weights).reshape(1, -1)
        similarity = cosine_similarity(profile_vector, content_matrix).ravel()

        scores = CONTENT_WEIGHT * similarity + POPULARITY_WEIGHT * product_popularity_scaled.to_numpy()
        reason = "Similar to items you've bought or liked"

    result = pd.DataFrame({
        "Product_ID": product_ids,
        "score": scores,
        "reason": reason,
    })
    result = result[candidate_mask].merge(catalog, on="Product_ID", how="left")

    return result.sort_values("score", ascending=False).head(top_n).reset_index(drop=True)


# Try it on a customer with history and a cold-start-ish one
sample_customer = interactions["Customer_ID"].iloc[0]
recommend_for_customer(sample_customer)[["Product_ID", "Product_Name", "score","reason"]]


,Product_ID,Product_Name,score,reason
0,P5123,Scandinavian Black Storage Cabinet,0.569327,Similar to items you've bought or liked
1,P5134,Scandinavian Grey Throw Pillow,0.540455,Similar to items you've bought or liked
2,P5121,Modern Sage Green Desk,0.525105,Similar to items you've bought or liked
3,P5115,Bohemian Terracotta Outdoor Sofa,0.515611,Similar to items you've bought or liked
4,P5086,Farmhouse Natural Sculpture,0.512494,Similar to items you've bought or liked


In [39]:
# 16. Sanity check the cold-start path for a customer with no interactions
recommend_for_customer("CUST_UNKNOWN")[["Product_ID", "Product_Name", "score", "reason"]]


,Product_ID,Product_Name,score,reason
0,P5042,Industrial Black Pillowcase Set,1.000000,Popular right now
1,P5077,Farmhouse Rust Bar Stool,0.951220,Popular right now
2,P5063,Traditional Ivory Sofa,0.926829,Popular right now
3,P5027,Traditional Blue Washcloth,0.853659,Popular right now
4,P5083,Farmhouse Rust Bar Stool,0.841463,Popular right now


In [46]:
import joblib
from pathlib import Path

#Package all trained components into a dictionary
trained_model_artifacts = {
    "content_transformer": content_transformer,      # The fitted encoder/scaler
    "interactions": interactions,                    # Customer purchase history
    "content_matrix": content_matrix,                # Crate & Barrel product features
    "product_id_to_idx": product_id_to_idx,          # Index map for Crate & Barrel
}
output_path = "../model/recommender_model.joblib"
joblib.dump(trained_model_artifacts, "../models/recommender_model.joblib")
print(f"Model successfully exported to {output_path}")

Model successfully exported to ../model/recommender_model.joblib
